# ChemML Mini-Lab #01: Can a Neural Network Learn to Count?
### A Toy Experiment on Out-of-Distribution Extrapolation & Physics-Informed Inductive Bias

**Author:** Amir Mostaghimi, Ph.D. ([amir-b-mostaghimi.github.io](https://amir-b-mostaghimi.github.io) · [ORCID](https://orcid.org/0000-0002-8084-9498))

**Question:** If we train standard neural networks (MLP, GRU, Transformer) to predict the next integer in variable-length counting sequences strictly within `[1, 50]`—e.g. `[1, 2, 3] → 4`, `[10, 11, 12, 13, 14] → 15`—can they extrapolate to `[100, 101, 102]` or `[9997, 9998, 9999]`?

And what happens when we inject one exact physical symmetry (**translation equivariance**: $f(x + c) = f(x) + c$)?

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

SEED = 42
MIN_LEN, MAX_LEN = 2, 6

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def make_dataset(n_samples, low, high, seed):
    rng = random.Random(seed)
    xs, masks, ys = [], [], []
    for _ in range(n_samples):
        length = rng.randint(MIN_LEN, MAX_LEN)
        start = rng.randint(low, high - length)
        seq = [float(start + i) for i in range(length)]
        pad_n = MAX_LEN - length
        xs.append([0.0] * pad_n + seq)
        masks.append([0.0] * pad_n + [1.0] * length)
        ys.append([float(start + length)])
    return (
        torch.tensor(xs, dtype=torch.float32),
        torch.tensor(masks, dtype=torch.float32),
        torch.tensor(ys, dtype=torch.float32),
    )

x_train, m_train, y_train = make_dataset(200, 1, 50, seed=100)
print('Training set shape:', x_train.shape, 'Example:', x_train[0].tolist(), '->', y_train[0].item())

## 1. Define Standard Models vs. Physics-Informed Translation-Equivariant Model
Notice that `PhysicsInformedCountingNet` has only **225 parameters** (vs. 5,000–9,000 for the standard models) and enforces exact translation equivariance by shifting coordinates relative to `x_last`.

In [ ]:
class StandardMLP(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(MAX_LEN * 2, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x, mask):
        return self.net(torch.cat([(x / 50.0) * mask, mask], dim=-1)) * 50.0

class CountingGRU(nn.Module):
    def __init__(self, hidden=48):
        super().__init__()
        self.gru = nn.GRU(2, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x, mask):
        inp = torch.stack([(x / 50.0) * mask, mask], dim=-1)
        out, _ = self.gru(inp)
        return self.head(out[:, -1, :]) * 50.0

class CountingTransformer(nn.Module):
    def __init__(self, d_model=32, nhead=4):
        super().__init__()
        self.in_proj = nn.Linear(2, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, MAX_LEN, d_model) * 0.05)
        layer = nn.TransformerEncoderLayer(d_model, nhead, 64, dropout=0.0, activation='gelu', batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
        self.head = nn.Linear(d_model, 1)
    def forward(self, x, mask):
        inp = torch.stack([(x / 50.0) * mask, mask], dim=-1)
        h = self.encoder(self.in_proj(inp) + self.pos_emb, src_key_padding_mask=(mask == 0))
        return self.head(h[:, -1, :]) * 50.0

class PhysicsInformedCountingNet(nn.Module):
    """Enforces exact translation equivariance: f(x + c) = f(x) + c."""
    def __init__(self, hidden=16):
        super().__init__()
        self.delta_net = nn.Sequential(nn.Linear(MAX_LEN * 2, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, x, mask):
        x_last = x[:, -1:]
        x_rel = (x - x_last) * mask
        return x_last + self.delta_net(torch.cat([x_rel, mask], dim=-1))

## 2. Train on `[1–50]` and Evaluate In-Distribution vs. Out-of-Distribution

In [ ]:
def train_model(model, epochs=450, lr=5e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(x_train, m_train), y_train)
        loss.backward()
        opt.step()
        sched.step()
    model.eval()
    return model

@torch.no_grad()
def eval_mae(model, ds):
    return float(torch.mean(torch.abs(model(ds[0], ds[1]) - ds[2])).item())

test_sets = {
    'ID [1-50]': make_dataset(100, 1, 50, seed=201),
    'Near OOD [60-150]': make_dataset(100, 60, 150, seed=202),
    'Far OOD [500-1000]': make_dataset(100, 500, 1000, seed=203),
    'Extreme OOD [5k-10k]': make_dataset(100, 5000, 10000, seed=204),
}

factories = {
    'MLP (ReLU)': StandardMLP,
    'GRU (RNN)': CountingGRU,
    'Transformer': CountingTransformer,
    'Physics-Informed Net': PhysicsInformedCountingNet,
}

trained = {}
for name, cls in factories.items():
    set_seed(SEED)
    m = train_model(cls())
    trained[name] = m
    n_params = sum(p.numel() for p in m.parameters())
    scores = ' | '.join(f'{k}: {eval_mae(m, ds):.4f}' for k, ds in test_sets.items())
    print(f'{name:22s} ({n_params:5d} params) -> {scores}')

## 3. Probe Concrete Sequences (`[18, 19, 20]` vs. `[9997, 9998, 9999]`)

In [ ]:
@torch.no_grad()
def predict_seq(model, seq):
    pad_n = MAX_LEN - len(seq)
    x = torch.tensor([[0.0] * pad_n + [float(v) for v in seq]])
    m = torch.tensor([[0.0] * pad_n + [1.0] * len(seq)])
    return float(model(x, m).item())

for seq in [[1, 2, 3], [10, 11, 12, 13, 14], [18, 19, 20], [100, 101, 102], [742, 743, 744, 745], [9997, 9998, 9999]]:
    preds = ' | '.join(f'{n}: {predict_seq(m, seq):.2f}' for n, m in trained.items())
    print(f'{str(seq):24s} (true={seq[-1]+1:5d}) -> {preds}')